We simulate a real streaming analytics pipeline: replay historical data at accelerated time, compute lifecycle-aware metrics in near real-time, store them in ClickHouse, and visualize in a professional dashboard.”

Streaming Workload Intelligence (Demo Mode)
Replay 1-day parquet at 60× speed (1 hour data = 1 minute real time), compute lifecycle-aware 5-min metrics separately for provisioned/serverless, store in ClickHouse with run isolation (run_id) for a live dashboard

In [ ]:
#1
!pip -q install clickhouse-connect pandas pyarrow
!pip -q install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.6 MB/s eta 0:00:00


In [ ]:
#2
#PARQUET_LOCAL_PATH = "/content/redshift_queries_2024-05-11_capped.parquet"

# Replay speed: 1 hour dataset time = 1 minute real time
SPEEDUP = 60.0

# Replay granularity: sleep once per slice
SLICE_DATASET_SECONDS = 10  # 10s dataset time => ~0.166s real time at 60×

BUCKET_MINUTES = 5

# Caps
CAP_QUEUE_MS   = 2 * 60 * 60 * 1000
CAP_EXEC_MS    = 24 * 60 * 60 * 1000
CAP_SCANNED_MB = 2_000_000.0
CAP_SPILLED_MB = 2_000_000.0

# Parquet streaming
PARQUET_BATCH_ROWS = 120_000   # tune for 12GB RAM (100k-200k)
CARRY_ROWS = 200_000           # ordering buffer; tune (150k-400k)

# Instance analytics
ENABLE_INSTANCE_TOPK = True
TOPK_INSTANCES_PER_BUCKET = 20

# ClickHouse insert batching
INSERT_EVERY_N_FLUSHES = 50    # fewer, bigger inserts

REQUIRED_COLS = [
    "arrival_timestamp",
    "deployment_type",
    "queue_duration_ms",
    "execution_duration_ms",
    "mbytes_scanned",
    "mbytes_spilled",
    "instance_id",
]



In [ ]:
#3
import uuid, datetime
RUN_ID = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]
print("✅ RUN_ID =", RUN_ID)



✅ RUN_ID = 20260201_123952_4295e804


/tmp/ipython-input-2709570411.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_ID = datetime.datetime.utcnow().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8]


In [ ]:
#4
import socket
print(socket.gethostbyname("wkixlqz135.eu-central-1.aws.clickhouse.cloud"))


63.177.172.19


In [ ]:
#5
import clickhouse_connect
CH = {
    "host": "wkixlqz135.eu-central-1.aws.clickhouse.cloud",
    "port": 8443,
    "username": "default",
    "password": "S1uOU_hkoUkDc",
    "secure": True,
}

client = clickhouse_connect.get_client(
    host=CH["host"],
    port=CH["port"],
    username=CH["username"],
    password=CH["password"],
    secure=CH["secure"],
)

print("Connected. Version:", client.query("SELECT version()").result_rows[0][0])


Connected. Version: 25.8.1.36912


In [ ]:
# ============================
# CELL 6: Create tables + MVs (system + instance)
# ============================

# 0) raw_events
client.command("""
CREATE TABLE IF NOT EXISTS raw_events (
  run_id String,
  arrival_timestamp DateTime64(6, 'UTC'),
  deployment_type LowCardinality(String),

  queue_duration_ms UInt32,
  execution_duration_ms UInt32,

  mbytes_scanned Float64,
  mbytes_spilled Float64,

  instance_id Nullable(UInt64),

  inserted_at DateTime('UTC') DEFAULT now()
)
ENGINE = MergeTree
PARTITION BY run_id
ORDER BY (run_id, arrival_timestamp, deployment_type)
""")

# 1) system metrics table (5-min)
client.command("""
CREATE TABLE IF NOT EXISTS system_metrics_5min (
  run_id String,
  bucket_start DateTime('UTC'),
  deployment_type LowCardinality(String),

  running_count UInt64,
  queued_count UInt64,

  queue_pressure Float64,
  spill_pressure Float64,
  pressure_level LowCardinality(String),

  throughput_mb_s Float64,

  inserted_at DateTime('UTC') DEFAULT now()
)
ENGINE = ReplacingMergeTree(inserted_at)
PARTITION BY run_id
ORDER BY (run_id, bucket_start, deployment_type)
""")

# 2) instance metrics table (5-min) ✅ includes inserted_at for dedupe
client.command("""
CREATE TABLE IF NOT EXISTS instance_topk_5min (
  run_id String,
  bucket_start DateTime('UTC'),
  deployment_type LowCardinality(String),
  instance_id UInt64,

  running_count UInt64,
  scanned_mb Float64,
  spilled_mb Float64,

  inserted_at DateTime('UTC') DEFAULT now()
)
ENGINE = ReplacingMergeTree(inserted_at)
PARTITION BY run_id
ORDER BY (run_id, bucket_start, deployment_type, instance_id)
""")

# Drop MVs if re-running
client.command("DROP VIEW IF EXISTS mv_system_metrics_5min")
client.command("DROP VIEW IF EXISTS mv_instance_metrics_5min")

# 3) System MV (5-min buckets)
client.command("""
CREATE MATERIALIZED VIEW mv_system_metrics_5min
TO system_metrics_5min
AS
WITH base AS (
  SELECT
    run_id,
    toStartOfInterval(toDateTime(arrival_timestamp), INTERVAL 5 MINUTE) AS bucket_start,
    deployment_type,

    -- simplified counts per bucket
    count() AS running_count,
    sum(queue_duration_ms > 0) AS queued_count,

    sum(mbytes_scanned) AS scanned_mb,
    sum(mbytes_spilled) AS spilled_mb,

    sum(execution_duration_ms) AS exec_ms
  FROM raw_events
  GROUP BY run_id, bucket_start, deployment_type
)
SELECT
  run_id,
  bucket_start,
  deployment_type,

  running_count,
  queued_count,

  if((queued_count + running_count) > 0,
     queued_count / (queued_count + running_count),
     0.0) AS queue_pressure,

  if((scanned_mb + spilled_mb) > 0,
     spilled_mb / (scanned_mb + spilled_mb),
     0.0) AS spill_pressure,

  multiIf(
    (spill_pressure > 0.6) OR (queue_pressure > 0.6), 'HIGH',
    (spill_pressure > 0.3) OR (queue_pressure > 0.3), 'MEDIUM',
    'LOW'
  ) AS pressure_level,

  if(exec_ms > 0, scanned_mb / (exec_ms / 1000.0), 0.0) AS throughput_mb_s,

  now() AS inserted_at
FROM base
""")

# 4) Instance MV (5-min buckets, per instance)
client.command("""
CREATE MATERIALIZED VIEW mv_instance_metrics_5min
TO instance_topk_5min
AS
SELECT
  run_id,
  toStartOfInterval(toDateTime(arrival_timestamp), INTERVAL 5 MINUTE) AS bucket_start,
  deployment_type,
  toUInt64(instance_id) AS instance_id,

  count() AS running_count,
  sum(mbytes_scanned) AS scanned_mb,
  sum(mbytes_spilled) AS spilled_mb,

  now() AS inserted_at
FROM raw_events
WHERE instance_id IS NOT NULL
GROUP BY run_id, bucket_start, deployment_type, instance_id""")


print("✅ Tables + MVs ready: raw_events, system_metrics_5min, instance_topk_5min")


✅ Tables + MVs ready: raw_events, system_metrics_5min, instance_topk_5min


In [ ]:
# ============================
# CELL 7: Utilization + Prediction (ClickHouse-side)
# ============================

# Dedup view (if duplicates exist, argMax keeps latest)
client.command("""
CREATE OR REPLACE VIEW instance_topk_5min_dedup AS
SELECT
  run_id,
  bucket_start,
  deployment_type,
  instance_id,
  argMax(running_count, inserted_at) AS running_count
FROM instance_topk_5min
GROUP BY run_id, bucket_start, deployment_type, instance_id
""")

# Capacity table (p95 running_count per instance)
client.command("""
CREATE TABLE IF NOT EXISTS instance_capacity_p95 (
  run_id String,
  deployment_type LowCardinality(String),
  instance_id UInt64,
  cap_p95 Float64,
  computed_at DateTime('UTC') DEFAULT now()
)
ENGINE = ReplacingMergeTree(computed_at)
ORDER BY (run_id, deployment_type, instance_id)
""")

# View: util + util_pred + residual
client.command("""
CREATE OR REPLACE VIEW instance_util_pred_5min AS
WITH base AS (
  SELECT
    d.run_id,
    d.bucket_start,
    d.deployment_type,
    d.instance_id,
    d.running_count,
    (d.running_count / greatest(c.cap_p95, 1)) AS util
  FROM instance_topk_5min_dedup d
  LEFT JOIN instance_capacity_p95 c
    ON d.run_id = c.run_id
   AND d.deployment_type = c.deployment_type
   AND d.instance_id = c.instance_id
)
SELECT
  *,
  median(util) OVER (
    PARTITION BY run_id, deployment_type, instance_id
    ORDER BY bucket_start
    ROWS BETWEEN 12 PRECEDING AND 1 PRECEDING
  ) AS util_pred,
  (util - util_pred) AS util_residual
FROM base
""")

# Helper: latest util for top-K instance selection
client.command("""
CREATE OR REPLACE VIEW instance_top_active AS
SELECT
  run_id,
  deployment_type,
  instance_id,
  argMax(util, bucket_start) AS util_latest
FROM instance_util_pred_5min
GROUP BY run_id, deployment_type, instance_id
""")

print("✅ Utilization + prediction views ready")


✅ Utilization + prediction views ready


In [ ]:
client.query("SHOW TABLES").result_rows


[('instance_capacity_p95',),
 ('instance_top_active',),
 ('instance_topk_5min',),
 ('instance_topk_5min_dedup',),
 ('instance_util_pred_5min',),
 ('latest_snapshot',),
 ('mv_instance_metrics_5min',),
 ('mv_system_metrics_5min',),
 ('raw_events',),
 ('system_metrics_5min',)]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
!cp "/content/drive/MyDrive/combined_2024-05-11.parquet" "/content/combined_2024-05-11.parquet"


In [ ]:
PARQUET_PATH = "/content/combined_2024-05-11.parquet"


In [ ]:
'''import os, subprocess

print("Exists:", os.path.exists(PARQUET_PATH))
print("Size (MB):", round(os.path.getsize(PARQUET_PATH) / (1024*1024), 2))

print("\nFile type:")
print(subprocess.check_output(["file", PARQUET_PATH]).decode())

print("\nFirst 200 bytes:")
print(subprocess.check_output(
    ["bash", "-lc", f"head -c 200 '{PARQUET_PATH}' | cat"]
).decode(errors="replace"))'''


'import os, subprocess\n\nprint("Exists:", os.path.exists(PARQUET_PATH))\nprint("Size (MB):", round(os.path.getsize(PARQUET_PATH) / (1024*1024), 2))\n\nprint("\nFile type:")\nprint(subprocess.check_output(["file", PARQUET_PATH]).decode())\n\nprint("\nFirst 200 bytes:")\nprint(subprocess.check_output(\n    ["bash", "-lc", f"head -c 200 \'{PARQUET_PATH}\' | cat"]\n).decode(errors="replace"))'

In [ ]:
#8

import duckdb
import pandas as pd

PARQUET_PATH = "/content/combined_2024-05-11.parquet"  # adjust if needed

con = duckdb.connect(database=":memory:")

NEEDED_COLS = [
    "arrival_timestamp",
    "deployment_type",
    "queue_duration_ms",
    "execution_duration_ms",
    "mbytes_scanned",
    "mbytes_spilled",
    "instance_id",
]

def iter_event_batches_duckdb(batch_rows=200_000):
    # DuckDB will handle weird parquet metadata better than pyarrow.dataset
    # Use ORDER BY arrival_timestamp so we avoid heavy sorts later
    query = f"""
    SELECT
      arrival_timestamp,
      lower(trim(deployment_type)) AS deployment_type,
      queue_duration_ms,
      execution_duration_ms,
      mbytes_scanned,
      mbytes_spilled,
      instance_id
    FROM read_parquet('{PARQUET_PATH}')
    WHERE deployment_type IS NOT NULL
    ORDER BY arrival_timestamp
    """
    rel = con.sql(query)

    offset = 0
    while True:
        batch = rel.limit(batch_rows, offset=offset).df()
        if len(batch) == 0:
            break

        batch["arrival_timestamp"] = pd.to_datetime(batch["arrival_timestamp"], utc=True, errors="coerce")
        batch = batch.dropna(subset=["arrival_timestamp"])
        batch = batch[batch["deployment_type"].isin(["provisioned", "serverless"])].copy()

        # numeric cleanup
        batch["queue_duration_ms"] = pd.to_numeric(batch["queue_duration_ms"], errors="coerce").fillna(0)
        batch["execution_duration_ms"] = pd.to_numeric(batch["execution_duration_ms"], errors="coerce").fillna(0)
        batch["mbytes_scanned"] = pd.to_numeric(batch["mbytes_scanned"], errors="coerce").fillna(0.0)
        batch["mbytes_spilled"] = pd.to_numeric(batch["mbytes_spilled"], errors="coerce").fillna(0.0)
        batch["instance_id"] = pd.to_numeric(batch["instance_id"], errors="coerce")

        yield batch
        offset += batch_rows


In [ ]:
#9
import time
import pandas as pd

SPEEDUP = 60.0
SLICE_DATASET_SECONDS = 60
BATCH_ROWS = 200_000

first_ts = None
wall_start = None

def sleep_to_match_speed(slice_end_ts):
    global first_ts, wall_start
    if first_ts is None:
        first_ts = slice_end_ts
        wall_start = time.time()
    dataset_s = (slice_end_ts - first_ts).total_seconds()
    target_wall = wall_start + dataset_s / SPEEDUP
    now = time.time()
    if target_wall > now:
        time.sleep(target_wall - now)

# Clean only this run_id
client.command(f"ALTER TABLE raw_events DELETE WHERE run_id = '{RUN_ID}'")
client.command(f"ALTER TABLE system_metrics_5min DELETE WHERE run_id = '{RUN_ID}'")
client.command(f"ALTER TABLE instance_topk_5min DELETE WHERE run_id = '{RUN_ID}'")
print("✅ Cleared this RUN_ID (async deletes)")

processed = 0
t0 = time.time()

carry = pd.DataFrame()

def normalize_for_raw_events(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "run_id": RUN_ID,
        "arrival_timestamp": df["arrival_timestamp"],
        "deployment_type": df["deployment_type"].astype(str).str.strip().str.lower(),

        "queue_duration_ms": (
            pd.to_numeric(df["queue_duration_ms"], errors="coerce")
            .fillna(0).clip(lower=0, upper=4_294_967_295).astype("uint32")
        ),
        "execution_duration_ms": (
            pd.to_numeric(df["execution_duration_ms"], errors="coerce")
            .fillna(0).clip(lower=0, upper=4_294_967_295).astype("uint32")
        ),

        "mbytes_scanned": pd.to_numeric(df["mbytes_scanned"], errors="coerce").fillna(0.0).clip(lower=0.0),
        "mbytes_spilled": pd.to_numeric(df["mbytes_spilled"], errors="coerce").fillna(0.0).clip(lower=0.0),

        "instance_id": pd.to_numeric(df["instance_id"], errors="coerce"),
    })

    out = out[out["deployment_type"].isin(["provisioned", "serverless"])].copy()
    out = out.dropna(subset=["arrival_timestamp"])

    # Nullable UInt64
    out["instance_id"] = out["instance_id"].where(out["instance_id"].notna(), None)
    out["instance_id"] = out["instance_id"].apply(lambda x: int(x) if x is not None else None)

    return out

print("✅ Starting live replay -> ClickHouse raw_events (MV computes metrics automatically)")

for chunk in iter_event_batches_duckdb(batch_rows=BATCH_ROWS):
    chunk = chunk.sort_values("arrival_timestamp").reset_index(drop=True)

    if len(carry) > 0:
        chunk = pd.concat([carry, chunk], ignore_index=True)
        chunk = chunk.sort_values("arrival_timestamp").reset_index(drop=True)
        carry = pd.DataFrame()

    i = 0
    n = len(chunk)
    while i < n:
        slice_start = chunk.loc[i, "arrival_timestamp"]
        slice_end = slice_start + pd.Timedelta(seconds=SLICE_DATASET_SECONDS)

        j = i
        while j < n and chunk.loc[j, "arrival_timestamp"] < slice_end:
            j += 1

        slice_df = chunk.iloc[i:j]
        raw_df = normalize_for_raw_events(slice_df)

        if len(raw_df) > 0:
            client.insert_df("raw_events", raw_df)
            processed += len(raw_df)

        sleep_to_match_speed(slice_end)

        if processed > 0 and processed % 500_000 == 0:
            print(f"inserted={processed:,} rows  wall={time.time()-t0:.1f}s")

        i = j

    if n > 0:
        carry = chunk.tail(1000).copy()

print("✅ Done inserting raw events")
print("total inserted:", f"{processed:,}")
print("wall seconds:", round(time.time() - t0, 2))


✅ Cleared this RUN_ID (async deletes)
✅ Starting live replay -> ClickHouse raw_events (MV computes metrics automatically)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Done inserting raw events
total inserted: 6,355,341
wall seconds: 1444.59


In [ ]:
#11
# ============================
# Capacity refresh for this RUN_ID
# Run anytime while replay is running
# ============================

client.command("ALTER TABLE instance_capacity_p95 DELETE WHERE run_id = %(run_id)s", parameters={"run_id": RUN_ID})

client.command("""
INSERT INTO instance_capacity_p95 (run_id, deployment_type, instance_id, cap_p95)
SELECT
  run_id,
  deployment_type,
  instance_id,
  greatest(quantileExact(0.95)(running_count), 1) AS cap_p95
FROM instance_topk_5min_dedup
WHERE run_id = %(run_id)s
GROUP BY run_id, deployment_type, instance_id
""", parameters={"run_id": RUN_ID})

print("✅ Capacity refreshed for:", RUN_ID)


In [ ]:
#monitor A
client.query(f"""
SELECT
  count() AS rows,
  min(bucket_start),
  max(bucket_start)
FROM system_metrics_5min
WHERE run_id = '{RUN_ID}'
""").result_rows


[(4868,
  datetime.datetime(2024, 5, 11, 0, 0),
  datetime.datetime(2024, 5, 11, 21, 15))]

In [ ]:
#monitor B
client.query(f"""
SELECT
  deployment_type,
  max(bucket_start) AS latest_bucket,
  argMax(queue_pressure, bucket_start) AS queue_p,
  argMax(spill_pressure, bucket_start) AS spill_p,
  argMax(pressure_level, bucket_start) AS level
FROM system_metrics_5min
WHERE run_id = '{RUN_ID}'
GROUP BY deployment_type
ORDER BY deployment_type
""").result_rows


[('provisioned',
  datetime.datetime(2024, 5, 11, 21, 15),
  0.0,
  0.01482637535902816,
  'LOW'),
 ('serverless', datetime.datetime(2024, 5, 11, 21, 15), 0.0, 0.0, 'LOW')]

In [ ]:
#10
q = f"""
SELECT
  run_id,
  bucket_start,
  deployment_type,
  running_count,
  queued_count,
  queue_pressure,
  spill_pressure,
  pressure_level,
  throughput_mb_s
FROM system_metrics_5min
WHERE run_id = '{RUN_ID}'
ORDER BY bucket_start DESC
LIMIT 10
"""
res = client.query(q)
print("rows returned:", len(res.result_rows))
res.result_rows[:5]



rows returned: 10


[('20260131_125217_bbf50708',
  datetime.datetime(2024, 5, 11, 21, 15),
  'serverless',
  21,
  0,
  0.0,
  0.0,
  'LOW',
  107.50943396226415),
 ('20260131_125217_bbf50708',
  datetime.datetime(2024, 5, 11, 21, 15),
  'serverless',
  1,
  0,
  0.0,
  0.0,
  'LOW',
  3654.5333333333333),
 ('20260131_125217_bbf50708',
  datetime.datetime(2024, 5, 11, 21, 15),
  'serverless',
  1,
  0,
  0.0,
  0.0,
  'LOW',
  0.0),
 ('20260131_125217_bbf50708',
  datetime.datetime(2024, 5, 11, 21, 15),
  'provisioned',
  565,
  55,
  0.08870967741935484,
  0.04899249113553739,
  'LOW',
  991.8988930741314),
 ('20260131_125217_bbf50708',
  datetime.datetime(2024, 5, 11, 21, 15),
  'provisioned',
  28,
  13,
  0.3170731707317073,
  0.059295917409588894,
  'MEDIUM',
  613.945584338315)]

In [ ]:
#11
q = f"""
SELECT
  bucket_start,
  deployment_type,
  running_count,
  queued_count,
  queue_pressure,
  spill_pressure,
  pressure_level,
  throughput_mb_s
FROM system_metrics_5min
WHERE run_id = '{RUN_ID}'
ORDER BY bucket_start DESC, deployment_type
LIMIT 2
"""
client.query(q).result_rows


[(datetime.datetime(2024, 5, 11, 21, 15),
  'provisioned',
  2,
  0,
  0.0,
  0.01482637535902816,
  'LOW',
  2262.598327952081),
 (datetime.datetime(2024, 5, 11, 21, 15),
  'provisioned',
  3,
  0,
  0.0,
  0.2261955084405743,
  'LOW',
  214.14635705429833)]

In [ ]:
import requests
r = requests.post(
    "https://wkixlqz135.eu-central-1.aws.clickhouse.cloud:8443",
    auth=("default", "S1uOU_hkoUkDc"),
    data="SELECT 1"
)
print(r.status_code, r.text[:200])


200 1

